<a href="https://colab.research.google.com/github/jejenfel/Phising-Detection---Multi-Agent-System-Project/blob/main/ProjectAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings("ignore")



In [4]:
# Step 1: Load the Dataset
from google.colab import files
uploaded = files.upload()

# Assuming the file name is 'phishing_url_website.csv'
data = pd.read_csv('phishing_url_website.csv')

Saving phishing_url_website.csv to phishing_url_website.csv


In [5]:
# Step 2: Data Cleaning
# Drop unnecessary columns
if 'ID' in data.columns:
    data = data.drop(columns=['ID'])  # Drop ID column if present

# Handling missing values
data = data.fillna(method='ffill')  # Forward fill missing values

In [6]:
# Step 3: Feature Engineering
# Extracting numerical and binary features
data['LengthOfURL'] = data['URL'].apply(len)
data['SpecialCharCount'] = data['URL'].apply(lambda x: sum([1 for char in x if char in "!@#$%^&*()"]))
data['ContainsIP'] = data['URL'].apply(lambda x: 1 if any(char.isdigit() for char in x.split('/')[0]) else 0)
data['IsHTTPS'] = data['URL'].apply(lambda x: 1 if x.startswith("https://") else 0)

# Encoding categorical variables (if any, e.g., TLD)
if 'TLD' in data.columns:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')  # Updated argument
    encoded_tld = pd.DataFrame(encoder.fit_transform(data[['TLD']]))
    encoded_tld.columns = encoder.get_feature_names_out(['TLD'])
    data = pd.concat([data, encoded_tld], axis=1).drop(columns=['TLD'])

In [7]:
# Step 4: Splitting Features and Labels
X = data.drop(columns=['label', 'URL'])  # Drop label and URL columns
y = data['label']  # Assuming 'Label' contains the target (0 for legitimate, 1 for phishing)

In [8]:
print(data.dtypes)


URL                            object
Domain                         object
URLSimilarityIndex            float64
NoOfOtherSpecialCharsInURL      int64
SpacialCharRatioInURL         float64
                               ...   
TLD_yt                        float64
TLD_za                        float64
TLD_zm                        float64
TLD_zone                      float64
TLD_zw                        float64
Length: 395, dtype: object


In [9]:
# Step 5: Normalize Features
# Drop remaining non-numeric columns before scaling
non_numeric_columns = ['URL', 'Domain', 'Title']  # Add other non-numeric columns if needed
X_numeric = X.drop(columns=non_numeric_columns, errors='ignore')

# Normalize numeric features
scaler = MinMaxScaler()
X_normalized = pd.DataFrame(scaler.fit_transform(X_numeric), columns=X_numeric.columns)

In [10]:
print(X_normalized.head())

   URLSimilarityIndex  NoOfOtherSpecialCharsInURL  SpacialCharRatioInURL  \
0                 1.0                    0.000000               0.098592   
1                 1.0                    0.090909               0.485915   
2                 1.0                    0.090909               0.359155   
3                 1.0                    0.000000               0.105634   
4                 1.0                    0.000000               0.154930   

   IsHTTPS  LineOfCode  DomainTitleMatchScore  URLTitleMatchScore  \
0      1.0    0.048780               0.000000            0.000000   
1      1.0    0.054045               0.555556            0.555556   
2      1.0    0.040797               0.466667            0.466667   
3      1.0    0.105984               0.000000            0.000000   
4      1.0    0.044920               1.000000            1.000000   

   IsResponsive  HasDescription  HasSocialNet  ...  TLD_ws  TLD_wtf  TLD_xyz  \
0           1.0             0.0           0.0  .

In [11]:
print(X_normalized.describe())


       URLSimilarityIndex  NoOfOtherSpecialCharsInURL  SpacialCharRatioInURL  \
count       176262.000000               176262.000000          176262.000000   
mean             0.808941                    0.082400               0.299491   
std              0.265591                    0.137552               0.204287   
min              0.000000                    0.000000               0.000000   
25%              0.617128                    0.000000               0.140845   
50%              1.000000                    0.000000               0.211268   
75%              1.000000                    0.090909               0.436620   
max              1.000000                    1.000000               1.000000   

             IsHTTPS     LineOfCode  DomainTitleMatchScore  \
count  176262.000000  176262.000000          176262.000000   
mean        0.790346       0.073659               0.517849   
std         0.407063       0.110283               0.496657   
min         0.000000       0.00

In [12]:
# Step 6: Handle Class Imbalance using SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_normalized, y)

In [13]:
# Step 7: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

In [14]:
print("Target (y) column preview:")
print(y.head())
print("Target column value counts:")
print(y.value_counts())

Target (y) column preview:
0    1
1    1
2    1
3    1
4    1
Name: label, dtype: int64
Target column value counts:
label
1    104225
0     72037
Name: count, dtype: int64


In [15]:
# Display final preprocessed data shapes
print("Training Features Shape:", X_train.shape)
print("Training Labels Shape:", y_train.shape)
print("Test Features Shape:", X_test.shape)
print("Test Labels Shape:", y_test.shape)

Training Features Shape: (166760, 391)
Training Labels Shape: (166760,)
Test Features Shape: (41690, 391)
Test Labels Shape: (41690,)


In [16]:
# Save preprocessed data for model training
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Preprocessing completed. Files saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv")


Preprocessing completed. Files saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv


In [17]:
# Import necessary libraries
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [18]:
# Define the Searching Agent
class FeatureSelector:
    def __init__(self):
        self.selected_features = None

    def select_features(self, X_train, y_train):
        """Select the most important features using a Random Forest model."""
        print("Performing Feature Selection...")
        model = RandomForestClassifier(random_state=42)
        model.fit(X_train, y_train)
        feature_importances = model.feature_importances_
        sorted_indices = np.argsort(feature_importances)[-5:]  # Select top 5 features
        self.selected_features = X_train.columns[sorted_indices]
        print("Selected Features:", self.selected_features.tolist())
        return self.selected_features


In [19]:

# Define the Reasoning Agent
class ReasoningAgent:
    def __init__(self):
        self.model = None

    def train_model(self, X_train, y_train):
        """Train a Random Forest model."""
        print("Training the Phishing Detection Model...")
        self.model = RandomForestClassifier(random_state=42)
        self.model.fit(X_train, y_train)
        print("Model Training Complete.")

    def evaluate_model(self, X_test, y_test):
        """Evaluate the model on the test set."""
        print("Evaluating the Model...")
        y_pred = self.model.predict(X_test)
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))
        print("Accuracy Score:", accuracy_score(y_test, y_pred))



In [20]:
# Define the Multi-Agent System
class MultiAgentSystem:
    def __init__(self, X_train, X_test, y_train, y_test):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.feature_selector = FeatureSelector()
        self.reasoning_agent = ReasoningAgent()

    def execute(self):
        # Step 1: Feature Selection
        selected_features = self.feature_selector.select_features(self.X_train, self.y_train)
        X_train_selected = self.X_train[selected_features]
        X_test_selected = self.X_test[selected_features]

        # Step 2: Model Training
        self.reasoning_agent.train_model(X_train_selected, self.y_train)

        # Step 3: Model Evaluation
        self.reasoning_agent.evaluate_model(X_test_selected, self.y_test)

In [21]:
# Main Execution
if __name__ == "__main__":
    # Assume X_train, X_test, y_train, y_test are already defined from previous preprocessing code
    # Uncomment the following lines if using in a separate script:
    # from google.colab import files
    # uploaded = files.upload()
    # dataset_path = list(uploaded.keys())[0]
    # X_train, X_test, y_train, y_test = data_loader(dataset_path)  # Use your preprocessing function

    # Initialize and execute the Multi-Agent System
    mas = MultiAgentSystem(X_train, X_test, y_train, y_test)
    mas.execute()


Performing Feature Selection...
Selected Features: ['HasCopyrightInfo', 'NoOfSelfRef', 'NoOfImage', 'LineOfCode', 'URLSimilarityIndex']
Training the Phishing Detection Model...
Model Training Complete.
Evaluating the Model...

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20919
           1       1.00      1.00      1.00     20771

    accuracy                           1.00     41690
   macro avg       1.00      1.00      1.00     41690
weighted avg       1.00      1.00      1.00     41690

Accuracy Score: 0.9999040537299112
